# Transfomer库
## Pipeline
将预处理 -> 模型推理 -> 后处理这一复杂的全流程封装成了一个黑盒
### 1.tokenizer
### 2.Model
### 3.Post-processor

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

checkpoint = r"d:\deeplearning\bert-base-chinese-local"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
text = "HuggingFace让生活变得简单"
inputs = tokenizer(text, return_tensors="pt")
print(inputs[("input_ids")])

outputs = model(**inputs)
print(outputs)#二分类任务原始得分

predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
print(predictions)#二分类任务概率分布


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1112.67it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: d:\deeplearning\bert-base-chinese-local
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from dif

tensor([[ 101,  100, 6375, 4495, 3833, 1359, 2533, 5042, 1296,  102]])
SequenceClassifierOutput(loss=None, logits=tensor([[0.0937, 0.3292]], grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)
tensor([[0.4414, 0.5586]], grad_fn=<SoftmaxBackward0>)


# datasets库

In [2]:
from datasets import load_dataset
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
dataset = load_dataset("rotten-tomatoes")
print(dataset)

'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1018)' thrown while requesting HEAD https://huggingface.co/datasets/rotten-tomatoes/resolve/main/README.md
Retrying in 1s [Retry 1/5].


RuntimeError: Cannot send a request, as the client has been closed.

# trainer

In [3]:
from transformers import TrainingArguments,Trainer,AutoModelForSequenceClassification,AutoTokenizer

## 准备组件
model_name = r"d:\deeplearning\bert-base-chinese-local"
tokenizer = AutoTokenizer.from_pretrained(model_name)
def tokenize_funtion(examples):
    return tokenizer(examples["text"],padding="max_length",truncation=True)
tokenized_datasets = dataset.map(tokenize_funtion,batched=True)
model = AutoModelForSequenceClassification.from_pretrained(model_name,num_labels=2)
small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(100)) 
small_eval_dataset = tokenized_datasets["validation"].shuffle(seed=42).select(range(100))

## 配置参数
training_args = TrainingArguments(
    output_dir="test_trainer",
    eval_strategy="epoch",
    num_train_epochs=1,
)

## 启动训练
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = small_train_dataset,
    test_dataset = small_eval_dataset,
)
trainer.train()


NameError: name 'dataset' is not defined

# Evaluate

In [ ]:
import numpy as np
import evaluate

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    compute_metrics=compute_metrics,
)
trainer.train()